In [11]:
import json
import re
import os
import glob
import pandas as pd
from pathlib import Path
import ast
from typing import Dict, List

In [4]:
id_pattern = re.compile(r'ID\s+(\d+)')

with open("eval_prompts.json", "r") as f:
    eval_prompts = json.load(f)

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

out = {}

for itype, scenarios in eval_prompts.items():
    if itype == "json":
        continue

    for scenario, items in scenarios.items():
    
        out[scenario] = []

        for item in items:
            prompt = item['prompt']
            ids = id_pattern.findall(prompt)
            print(ids)
            out[scenario].append(ids)

['40679', '27706', '23132', '79411', '31113']
['58616', '41277', '66935', '49556', '30721']
['43363', '62549', '78147', '9795', '40487']
['6777', '24364', '71082', '36396', '19346']
['10825', '12421', '66807', '17550', '12376']
['62071', '73827', '25711', '23274', '3282']
['62609', '27706', '63485', '1268', '51114']
['27821', '20968', '37955', '40111', '1181']
['10984', '72266', '66807', '52457', '15613']
['63485', '35725', '2004', '48780', '51114']
['75828', '64781', '61767', '24364', '52457']
['12421', '53652', '10825', '12376', '47834']
['3282', '73827', '25711', '62071', '72346']
['3384', '7489', '13497', '25607', '20839']
['39691', '40877', '46039', '28694', '77515']
['74418', '31926', '48780', '37989', '60915']
['30004', '33749', '22002', '10131', '22487']
['12888', '5222', '60915', '77515', '70239']
['55950', '21765', '50181', '57316', '52548']
['77515', '50221', '7913', '72055', '33045']
['39691', '40877', '28694', '46039', '63902']
['47247', '74532', '38478', '26994', '33749']

In [ ]:

with open('groups_same_first.json', 'r') as f:
    groups = json.load(f)

with open('outputs/gemma/description__same_first.json', 'r') as f:
    gemma_desc = json.load(f)

with open('outputs/llama/description__same_first.json', 'r') as f:
    llama_desc = json.load(f)

def extract_id(pred_str):
    nums = re.findall(r'\d+', pred_str)
    return nums[-1] if nums else None

for idx, entry in enumerate(groups):
    entry['gemma_description_pick'] = extract_id(gemma_desc[idx]['predicted_id'])
    entry['llama_description_pick'] = extract_id(llama_desc[idx]['predicted_id'])

output_path = 'groups_same_first_results.json'
with open(output_path, 'w') as f:
    json.dump(groups, f, indent=2)

df = pd.DataFrame(groups)

df.to_csv("groups_same_first_results.csv", index=False)

In [28]:


with open('groups_diff_first.json', 'r') as f:
    groups = json.load(f)

with open('outputs/gemma/description__diff_first.json', 'r') as f:
    gemma_desc = json.load(f)

with open('outputs/llama/description__diff_first.json', 'r') as f:
    llama_desc = json.load(f)

def extract_id(pred_str):
    nums = re.findall(r'\d+', pred_str)
    return nums[-1] if nums else None

for idx, entry in enumerate(groups):
    entry['gemma_description_pick'] = extract_id(gemma_desc[idx]['predicted_id'])
    entry['llama_description_pick'] = extract_id(llama_desc[idx]['predicted_id'])

output_path = 'groups_diff_first_results.json'
with open(output_path, 'w') as f:
    json.dump(groups, f, indent=2)

df = pd.DataFrame(groups)

df.to_csv("groups_diff_first_results.csv", index=False)

In [57]:
# with open('groups_diff_first_results.json', 'r') as f:
#     data = json.load(f)

# df = pd.DataFrame(data)

df = pd.read_csv("groups_diff_first_results.csv")

gemma_correct = (df['gemma_description_pick'].astype(int) == df['outlier_id'].astype(int)).sum()
llama_correct = (df['llama_description_pick'].astype(int) == df['outlier_id'].astype(int)).sum()
op1_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[0]).astype(int) == df['outlier_id'].astype(int)).sum()
op2_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[1]).astype(int) == df['outlier_id'].astype(int)).sum()
op3_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[2]).astype(int) == df['outlier_id'].astype(int)).sum()
op4_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[3]).astype(int) == df['outlier_id'].astype(int)).sum()
last_correct = (df['last_option'].astype(int) == df['outlier_id'].astype(int)).sum()
total = len(df)

gemma_accuracy = gemma_correct / total
llama_accuracy = llama_correct / total
op1_accuracy = op1_correct / total
op2_accuracy = op2_correct / total
op3_accuracy = op3_correct / total
op4_accuracy = op4_correct / total
last_accuracy = last_correct / total

print(f"Gemma Accuracy: {gemma_correct}/{total} = {gemma_accuracy:.2%}")
print(f"Llama Accuracy: {llama_correct}/{total} = {llama_accuracy:.2%}")
print(f"op1 Accuracy: {op1_correct}/{total} = {op1_accuracy:.2%}")
print(f"op2 Accuracy: {op2_correct}/{total} = {op2_accuracy:.2%}")
print(f"op3 Accuracy: {op3_correct}/{total} = {op3_accuracy:.2%}")
print(f"op4 Accuracy: {op4_correct}/{total} = {op4_accuracy:.2%}")
print(f"Last Accuracy: {last_correct}/{total} = {last_accuracy:.2%}")

Gemma Accuracy: 24/100 = 24.00%
Llama Accuracy: 31/100 = 31.00%
op1 Accuracy: 18/100 = 18.00%
op2 Accuracy: 12/100 = 12.00%
op3 Accuracy: 19/100 = 19.00%
op4 Accuracy: 28/100 = 28.00%
Last Accuracy: 23/100 = 23.00%


In [58]:
# with open('groups_same_second_results.json', 'r') as f:
#     data = json.load(f)

# df = pd.DataFrame(data)

df = pd.read_csv("groups_same_second_results.csv")

gemma_correct = (df['gemma_description_pick'].astype(int) == df['outlier_id'].astype(int)).sum()
llama_correct = (df['llama_description_pick'].astype(int) == df['outlier_id'].astype(int)).sum()
op1_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[0]).astype(int) == df['outlier_id'].astype(int)).sum()
op2_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[1]).astype(int) == df['outlier_id'].astype(int)).sum()
op3_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[2]).astype(int) == df['outlier_id'].astype(int)).sum()
op4_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[3]).astype(int) == df['outlier_id'].astype(int)).sum()
last_correct = (df['last_option'].astype(int) == df['outlier_id'].astype(int)).sum()
total = len(df)

gemma_accuracy = gemma_correct / total
llama_accuracy = llama_correct / total
op1_accuracy = op1_correct / total
op2_accuracy = op2_correct / total
op3_accuracy = op3_correct / total
op4_accuracy = op4_correct / total
last_accuracy = last_correct / total

print(f"Gemma Accuracy: {gemma_correct}/{total} = {gemma_accuracy:.2%}")
print(f"Llama Accuracy: {llama_correct}/{total} = {llama_accuracy:.2%}")
print(f"op1 Accuracy: {op1_correct}/{total} = {op1_accuracy:.2%}")
print(f"op2 Accuracy: {op2_correct}/{total} = {op2_accuracy:.2%}")
print(f"op3 Accuracy: {op3_correct}/{total} = {op3_accuracy:.2%}")
print(f"op4 Accuracy: {op4_correct}/{total} = {op4_accuracy:.2%}")
print(f"Last Accuracy: {last_correct}/{total} = {last_accuracy:.2%}")

Gemma Accuracy: 26/100 = 26.00%
Llama Accuracy: 37/100 = 37.00%
op1 Accuracy: 16/100 = 16.00%
op2 Accuracy: 22/100 = 22.00%
op3 Accuracy: 14/100 = 14.00%
op4 Accuracy: 22/100 = 22.00%
Last Accuracy: 26/100 = 26.00%


In [59]:
df = pd.read_csv("groups_same_first_results.csv")

gemma_correct = (df['gemma_description_pick'].astype(int) == df['outlier_id'].astype(int)).sum()
llama_correct = (df['llama_description_pick'].astype(int) == df['outlier_id'].astype(int)).sum()
op1_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[0]).astype(int) == df['outlier_id'].astype(int)).sum()
op2_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[1]).astype(int) == df['outlier_id'].astype(int)).sum()
op3_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[2]).astype(int) == df['outlier_id'].astype(int)).sum()
op4_correct = (df['options'].apply(lambda s: ast.literal_eval(s)[3]).astype(int) == df['outlier_id'].astype(int)).sum()
last_correct = (df['last_option'].astype(int) == df['outlier_id'].astype(int)).sum()
total = len(df)

gemma_accuracy = gemma_correct / total
llama_accuracy = llama_correct / total
op1_accuracy = op1_correct / total
op2_accuracy = op2_correct / total
op3_accuracy = op3_correct / total
op4_accuracy = op4_correct / total
last_accuracy = last_correct / total

print(f"Gemma Accuracy: {gemma_correct}/{total} = {gemma_accuracy:.2%}")
print(f"Llama Accuracy: {llama_correct}/{total} = {llama_accuracy:.2%}")
print(f"op1 Accuracy: {op1_correct}/{total} = {op1_accuracy:.2%}")
print(f"op2 Accuracy: {op2_correct}/{total} = {op2_accuracy:.2%}")
print(f"op3 Accuracy: {op3_correct}/{total} = {op3_accuracy:.2%}")
print(f"op4 Accuracy: {op4_correct}/{total} = {op4_accuracy:.2%}")
print(f"Last Accuracy: {last_correct}/{total} = {last_accuracy:.2%}")

Gemma Accuracy: 20/100 = 20.00%
Llama Accuracy: 19/100 = 19.00%
op1 Accuracy: 25/100 = 25.00%
op2 Accuracy: 19/100 = 19.00%
op3 Accuracy: 17/100 = 17.00%
op4 Accuracy: 20/100 = 20.00%
Last Accuracy: 19/100 = 19.00%


In [5]:
import pandas as pd
import ast

In [6]:
def resolve_predicted_id(row):
    idx = row["pred_idx"]
    if 1 <= idx <= 5:
        # parse the options string into a Python list
        opts = ast.literal_eval(row["options"])
        # grab the (idx-1)th element
        return int(opts[idx - 1])
    else:
        # otherwise just return the numeric value itself
        return idx

In [15]:
types = ["same_second", "diff_first"]
models = ["vlm_qwen2.5-vl", "vlm_phi4_multimodal"]
folders = ["original_wlegend", "original", "reoriented", "reoriented_wlegend"]
modalities = ["image", "2modality"]
ref_df = pd.read_csv("./groups_{}_results.csv".format(type))

In [49]:
processed = [
    "./outputs/image/original_wlegend/vlm_qwen2.5-vl_same_second_wlegend.csv",
    "./outputs/2modality/original_wlegend/vlm_qwen2.5-vl_2modal_same_second_wlegend.csv",
    "./outputs/image/original/vlm_qwen2.5-vl_same_second.csv",
    "./outputs/2modality/original/vlm_qwen2.5-vl_2modal_same_second.csv",
    "./outputs/image/reoriented/vlm_qwen2.5-vl_same_second.csv",
    "./outputs/2modality/reoriented/vlm_qwen2.5-vl_2modal_same_second.csv",
    "./outputs/image/reoriented_wlegend/vlm_qwen2.5-vl_same_second_wlegend.csv",
    "./outputs/2modality/reoriented_wlegend/vlm_qwen2.5-vl_2modal_same_second_wlegend.csv",
    "./outputs/image/original_wlegend/vlm_phi4_multimodal_same_second_wlegend.csv",
    "./outputs/2modality/original_wlegend/vlm_phi4_multimodal_2modal_same_second_wlegend.csv",
    "./outputs/image/original/vlm_phi4_multimodal_same_second.csv",
    "./outputs/2modality/original/vlm_phi4_multimodal_2modal_same_second.csv",
    "./outputs/image/reoriented/vlm_phi4_multimodal_same_second.csv",
    "./outputs/2modality/reoriented/vlm_phi4_multimodal_2modal_same_second.csv",
    "./outputs/image/reoriented_wlegend/vlm_phi4_multimodal_same_second_wlegend.csv",
    "./outputs/2modality/reoriented_wlegend/vlm_phi4_multimodal_2modal_same_second_wlegend.csv",
    "./outputs/image/original_wlegend/vlm_qwen2.5-vl_diff_first_wlegend.csv",
    "./outputs/2modality/original_wlegend/vlm_qwen2.5-vl_2modal_diff_first_wlegend.csv",
    "./outputs/image/original/vlm_qwen2.5-vl_diff_first.csv",
    "./outputs/2modality/original/vlm_qwen2.5-vl_2modal_diff_first.csv",
    "./outputs/image/reoriented/vlm_qwen2.5-vl_diff_first.csv",
    "./outputs/2modality/reoriented/vlm_qwen2.5-vl_2modal_diff_first.csv",
    "./outputs/image/reoriented_wlegend/vlm_qwen2.5-vl_diff_first_wlegend.csv",
    "./outputs/2modality/reoriented_wlegend/vlm_qwen2.5-vl_2modal_diff_first_wlegend.csv",
    "./outputs/image/original_wlegend/vlm_phi4_multimodal_diff_first_wlegend.csv",
    "./outputs/2modality/original_wlegend/vlm_phi4_multimodal_2modal_diff_first_wlegend.csv",
    "./outputs/image/original/vlm_phi4_multimodal_diff_first.csv",
    "./outputs/2modality/original/vlm_phi4_multimodal_2modal_diff_first.csv",
    "./outputs/image/reoriented/vlm_phi4_multimodal_diff_first.csv"
]

In [50]:
for type in types:
    for model in models:
        for folder in folders:
            for modality in modalities:
                ref_df = pd.read_csv("./groups_{}_results.csv".format(type))
                suffix = ""
                infix = ""

                if "wlegend" in folder:
                    suffix = "_wlegend"

                if modality == "2modality":
                    infix = "2modal_"

                path = "./outputs/{}/{}/{}_{}{}{}.csv".format(modality, folder, model, infix, type, suffix)
                if path in processed:
                    continue
                print(path)
                # bad_rows = df[df.isnull().any(axis=1)]
                # print("Rows with at least one NaN:\n", bad_rows)
                df = pd.read_csv(path)
               
                df["options"] = ref_df["options"]
                df["last_option"] = ref_df["last_option"]

                

                nan_rows = df[df["options"].isna()]
                print("Rows with options == NaN:\n", nan_rows)

                df["pred_idx"] = df["predicted_id"].str.extract(r"(\d+)").astype(int)
                df["predicted_id"] = df.apply(resolve_predicted_id, axis=1)
                df = df.drop(columns="pred_idx")

                df.to_csv(path, index=False)


./outputs/2modality/reoriented/vlm_phi4_multimodal_2modal_diff_first.csv
Rows with options == NaN:
 Empty DataFrame
Columns: [base_cluster, other_cluster, options, outlier_id, predicted_id, raw_response, last_option]
Index: []
./outputs/image/reoriented_wlegend/vlm_phi4_multimodal_diff_first_wlegend.csv
Rows with options == NaN:
 Empty DataFrame
Columns: [base_cluster, other_cluster, options, outlier_id, predicted_id, raw_response, last_option]
Index: []
./outputs/2modality/reoriented_wlegend/vlm_phi4_multimodal_2modal_diff_first_wlegend.csv
Rows with options == NaN:
 Empty DataFrame
Columns: [base_cluster, other_cluster, options, outlier_id, predicted_id, raw_response, last_option]
Index: []


In [9]:
df = pd.read_csv("./outputs/{}/{}/{}_{}_wlegend.csv".format(modality, folder, model, type))

In [55]:
data = []

for type in types:
    for model in models:
        for folder in folders:
            for modality in modalities:
                ref_df = pd.read_csv("./groups_{}_results.csv".format(type))
                suffix = ""
                infix = ""
                legend = 0
                reoriented = 0

                if "wlegend" in folder:
                    legend = 1
                    suffix = "_wlegend"

                if modality == "2modality":
                    infix = "2modal_"

                if 'reoriented' in folder:
                    reoriented = 1

                path = "./outputs/{}/{}/{}_{}{}{}.csv".format(modality, folder, model, infix, type, suffix)
                df = pd.read_csv(path)

                # --- FIX: parse options as ints ---
                df['options_list'] = df['options'].apply(
                    lambda x: [int(opt) for opt in ast.literal_eval(x)]
                )

                # model accuracy
                df['model_correct'] = df['predicted_id'] == df['outlier_id']
                model_acc = df['model_correct'].mean()

                # baseline accuracies
                baseline_accs = {
                    f'opt{r+1}_acc': (
                        df['options_list']
                          .apply(lambda opts: opts[r] if len(opts)>r else None)
                          == df['outlier_id']
                    ).mean()
                    for r in range(5)
                }

                # pick‐rank frequencies
                def get_rank(pred, opts):
                    try:
                        return opts.index(pred) + 1
                    except ValueError:
                        return None

                df['pred_choice'] = df.apply(
                    lambda row: get_rank(row['predicted_id'], row['options_list']), axis=1
                )
                freq = df['pred_choice'].value_counts(normalize=True) \
                          .reindex(range(1,6), fill_value=0)
                freq_dict = {f'choice_{i}_freq': freq[i] for i in range(1,6)}

                # assemble row
                row = {
                    'task': type,
                    'model': model,
                    'modality': modality,
                    'reoriented': reoriented,
                    'legend': legend,
                    'model_accuracy': model_acc,
                    **baseline_accs,
                    **freq_dict
                }
                data.append(row)
# for type in types:
#     for model in models:
#         for folder in folders:
#             for modality in modalities:
#                 ref_df = pd.read_csv("./groups_{}_results.csv".format(type))
#                 suffix = ""
#                 infix = ""
#                 legend = 0

#                 if "wlegend" in folder:
#                     legend = 1
#                     suffix = "_wlegend"

#                 if modality == "2modality":
#                     infix = "2modal_"

#                 path = "./outputs/{}/{}/{}_{}{}{}.csv".format(modality, folder, model, infix, type, suffix)
#                 df = pd.read_csv(path)
#                 df['options_list'] = df['options'].apply(lambda x: ast.literal_eval(x))

#                 # compute model accuracy
#                 df['model_correct'] = df['predicted_id'] == df['outlier_id']
#                 model_acc = df['model_correct'].mean()

#                 # compute baseline accuracies for each rank
#                 baseline_accs = {}
#                 for rank in range(5):
#                     baseline_accs[f'first{rank+1}_acc'] = (df['options_list'].apply(lambda opts: opts[rank]) == df['outlier_id']).mean()

#                 # compute frequency of model picking each rank
#                 def get_rank(pred, opts):
#                     try:
#                         return opts.index(pred) + 1
#                     except ValueError:
#                         return None

#                 df['pred_rank'] = df.apply(lambda row: get_rank(row['predicted_id'], row['options_list']), axis=1)
#                 freq = df['pred_rank'].value_counts(normalize=True).reindex(range(1, 6), fill_value=0)
#                 freq_dict = {f'rank_{i}_freq': freq[i] for i in range(1, 6)}

#                 # assemble metadata and metrics
#                 row = {
#                     'task': type,
#                     'model': model,
#                     'modality': modality,
#                     'legend': legend,
#                     'model_accuracy': model_acc,
#                     **baseline_accs,
#                     **freq_dict
#                 }
#                 data.append(row)

# create summary DataFrame
summary_df = pd.DataFrame(data)

# display or save to CSV
print(summary_df)
summary_df.to_csv('summary_metrics.csv', index=False)

           task                model   modality  reoriented  legend  \
0   same_second       vlm_qwen2.5-vl      image           0       1   
1   same_second       vlm_qwen2.5-vl  2modality           0       1   
2   same_second       vlm_qwen2.5-vl      image           0       0   
3   same_second       vlm_qwen2.5-vl  2modality           0       0   
4   same_second       vlm_qwen2.5-vl      image           1       0   
5   same_second       vlm_qwen2.5-vl  2modality           1       0   
6   same_second       vlm_qwen2.5-vl      image           1       1   
7   same_second       vlm_qwen2.5-vl  2modality           1       1   
8   same_second  vlm_phi4_multimodal      image           0       1   
9   same_second  vlm_phi4_multimodal  2modality           0       1   
10  same_second  vlm_phi4_multimodal      image           0       0   
11  same_second  vlm_phi4_multimodal  2modality           0       0   
12  same_second  vlm_phi4_multimodal      image           1       0   
13  sa

In [10]:
df["options"] = ref_df["options"]
df["last_option"] = ref_df["last_option"]

df["pred_idx"] = df["predicted_id"].str.extract(r"(\d+)").astype(int)
df["predicted_id"] = df.apply(resolve_predicted_id, axis=1)
df = df.drop(columns="pred_idx")


In [11]:
df.to_csv("./outputs/{}/{}/{}_{}_wlegend.csv".format(modality, folder, model, type), index=False)

In [47]:
# 1. Load each CSV
df_phi = pd.read_csv("./outputs/image/{}/vlm_phi4_multimodal_{}_wlegend.csv".format(folder, type))
df_qwen = pd.read_csv("./outputs/image/{}/vlm_qwen2.5-vl_{}_wlegend.csv".format(folder, type))

# 2. Rename the predicted_id column and drop raw_response
df_phi = (
    df_phi
    .rename(columns={"predicted_id": "phi_pick"})
    .drop(columns=["raw_response"])
)
df_qwen = (
    df_qwen
    .rename(columns={"predicted_id": "qwen_pick"})
    .drop(columns=["raw_response"])
)

# 3. Merge on the common columns
common_cols = ["base_cluster", "other_cluster", "options", "outlier_id", "last_option"]
combined = pd.merge(df_phi, df_qwen, on=common_cols, how="outer")

# 4. (Optional) Reorder columns if you like
cols = common_cols + ["phi_pick", "qwen_pick"]
combined = combined[cols]

In [48]:
combined.to_csv("./outputs/image/{}/groups_{}_results.csv".format(folder, type), index=False)

In [2]:
LABEL_TO_INDEX = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
RE_PRED = re.compile(r'([A-E])')  # handles "** D", "E", etc.

def _parse_filename_meta(path: str) -> Dict[str, object]:
    """
    Parse filename into:
      - model: everything before the first '_' (e.g., 'vlm_qwen2.5-vl')
      - modality: 'img' or 'multi'
      - task: tokens between modality and orientation, joined with '_' (e.g., 'diff_first')
      - reoriented: 1 if 'reoriented', else 0 if 'original'
      - legend: 1 if 'wlegend', else 0 if 'nolegend'
    Works even if the middle section has variable underscores.
    """
    fname = os.path.basename(path)
    stem = fname[:-4] if fname.lower().endswith('.csv') else fname

    # model is everything before the first underscore
    if '_' not in stem:
        raise ValueError(f"Filename missing '_' separators: {fname}")
    model, rest = stem.split('_', 1)

    tokens: List[str] = rest.split('_')
    if len(tokens) < 4:
        raise ValueError(f"Filename '{fname}' does not have enough tokens for metadata.")

    # legend + orientation from the right
    legend_token = tokens[-1].lower()
    orient_token = tokens[-2].lower()
    if legend_token not in ('wlegend', 'nolegend'):
        raise ValueError(f"Expected legend token at end (wlegend/nolegend), got '{legend_token}' in {fname}")
    if orient_token not in ('reoriented', 'original'):
        raise ValueError(f"Expected orientation token before legend (reoriented/original), got '{orient_token}' in {fname}")

    # find modality by scanning leftwards for 'img' or 'multi'
    modality_idx = None
    for i in range(len(tokens) - 3, -1, -1):  # skip the last 2 (orient, legend) + at least one for task
        if tokens[i].lower() in ('img', 'multi'):
            modality_idx = i
            break
    if modality_idx is None:
        raise ValueError(f"Could not find modality ('img' or 'multi') in {fname}")

    modality = tokens[modality_idx].lower()

    # task = everything between modality and orientation (join with '_')
    task_tokens = tokens[modality_idx + 1 : -2]
    if not task_tokens:
        raise ValueError(f"No task tokens found between modality and orientation in {fname}")
    task = "_".join(task_tokens)  # e.g., "diff_first" or "same_second"

    # flags
    reoriented = 1 if orient_token == 'reoriented' else 0
    legend = 1 if legend_token == 'wlegend' else 0

    return {
        'model': model,
        'modality': modality,
        'task': task,
        'reoriented': reoriented,
        'legend': legend,
    }

def process_file_to_row(file_path: str) -> pd.DataFrame:
    """
    Load one CSV and return a single-row DataFrame with:
      model, modality, task, reoriented, legend, accuracy,
      count_A, count_B, count_C, count_D, count_E, n_examples, source_file.
    """
    df = pd.read_csv(file_path)

    # 1) predicted label (A–E) and chosen_id from options
    labels = df['predicted_id'].astype(str).str.extract(RE_PRED)[0]
    if labels.isna().any():
        bad = df['predicted_id'][labels.isna()]
        raise ValueError(f"Could not extract label A–E from some rows: {bad.tolist()[:5]}")

    # parse 'options' safely -> list
    options_list = df['options'].apply(ast.literal_eval)

    # pick the chosen ID per row
    chosen_ids = [opts[LABEL_TO_INDEX[lbl]] for opts, lbl in zip(options_list, labels)]

    # 2) accuracy vs outlier_id (compare as strings to avoid type mismatch)
    correct = (pd.Series(chosen_ids, index=df.index).astype(str) == df['outlier_id'].astype(str))
    accuracy = float(correct.mean())

    # 3) label counts
    counts = labels.value_counts().reindex(['A', 'B', 'C', 'D', 'E'], fill_value=0)

    # 4) filename metadata
    meta = _parse_filename_meta(file_path)

    # 5) one summary row
    row = {
        **meta,
        'accuracy': accuracy,
        'count_A': int(counts['A']),
        'count_B': int(counts['B']),
        'count_C': int(counts['C']),
        'count_D': int(counts['D']),
        'count_E': int(counts['E']),
        'n_examples': int(len(df)),
        'source_file': os.path.basename(file_path),
    }
    return pd.DataFrame([row])

def process_many_to_table(directory: str) -> pd.DataFrame:
    """
    Process every .csv in `directory` (non-recursive), returning a table
    with one summary row per file.
    """
    if not os.path.isdir(directory):
        raise NotADirectoryError(f"Not a directory: {directory}")

    files = sorted(glob.glob(os.path.join(directory, "*.csv")))
    if not files:
        raise FileNotFoundError(f"No .csv files found in: {directory}")

    rows = [process_file_to_row(f) for f in files]
    return pd.concat(rows, ignore_index=True)


NameError: name 're' is not defined

In [1]:
results = process_many_to_table("./outputs/vlm_qwen2.5-vl_w_door_bound_legendbound")
results

NameError: name 'process_many_to_table' is not defined